In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


In [2]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/othello-world_eval'

for root, dirs, files in os.walk(repo_path):
    # Skip hidden directories and common non-essential directories
    dirs[:] = [d for d in dirs if not d.startswith('.') and d not in ['__pycache__', '.git', 'node_modules']]
    
    level = root.replace(repo_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    
    subindent = ' ' * 2 * (level + 1)
    for file in files[:20]:  # Limit files shown per directory
        print(f'{subindent}{file}')
    if len(files) > 20:
        print(f'{subindent}... and {len(files) - 20} more files')

othello-world_eval/
  documentation.pdf
  plan.md
  .gitignore
  Othello_GPT_Circuits.ipynb
  environment.yml
  intervening_probe_interact_column.ipynb
  train_gpt_othello.ipynb
  CodeWalkthrough.md
  ckpts_synthetic_model.pth
  plot_attribution_via_intervention_othello.ipynb
  LICENSE
  train_probe_othello.py
  intervention_benchmark.pkl
  produce_probes.sh
  ckpts/
    ckpts/
      battery_othello/
        state_championship/
          layer0/
            tensorboard.txt
            checkpoint.ckpt
          layer7/
            tensorboard.txt
            checkpoint.ckpt
          layer3/
            checkpoint.ckpt
            tensorboard.txt
          layer4/
            tensorboard.txt
            checkpoint.ckpt
          layer6/
            tensorboard.txt
            checkpoint.ckpt
          layer1/
            checkpoint.ckpt
            tensorboard.txt
          layer5/
            checkpoint.ckpt
            tensorboard.txt
          layer2/
            tensorboard.txt
    

# Generalizability Evaluation for Othello-World

This notebook evaluates whether the findings in the Othello-World repository generalize beyond the original experimental setting.

## Repository Structure Overview

The repository contains:
- Main notebooks for circuit analysis (Othello_GPT_Circuits.ipynb, intervening_probe_interact_column.ipynb)
- Model checkpoints (ckpts_synthetic_model.pth, ckpts/)
- Probes for different layers
- Mechanistic interpretability utilities
- Existing evaluation folder with consistency and replication evaluations

In [3]:
# Check for existing evaluation files to understand what was done
import json

# Check the existing replication evaluation
try:
    with open('/net/scratch2/smallyan/othello-world_eval/evaluation/replications/self_replication_evaluation.json', 'r') as f:
        replication_eval = json.load(f)
    print("Replication Evaluation:")
    print(json.dumps(replication_eval, indent=2))
except Exception as e:
    print(f"Could not load replication eval: {e}")

Replication Evaluation:
{
  "Checklist": {
    "RP1_ImplementationReconstructability": "PASS",
    "RP2_EnvironmentReproducibility": "PASS",
    "RP3_DeterminismAndStability": "PASS"
  },
  "Rationale": {
    "RP1_ImplementationReconstructability": "The experiment can be fully reconstructed from plan.md and CodeWalkthrough.md. The plan specifies model architecture (8-layer GPT, 512-dim, 8 heads), training data (synthetic/championship datasets), methodology (probes, interventions), and expected metrics. No significant guesswork was required.",
    "RP2_EnvironmentReproducibility": "Environment fully restorable via environment.yml, pre-trained weights from HuggingFace, pre-computed game sequences and probes in repository, and TransformerLens library for model loading. Minor disk quota issue resolved by using alternative cache directory.",
    "RP3_DeterminismAndStability": "Results are stable across runs. Inference-only experiment with no training, fixed random seeds (42), pre-computed g

## Understanding the Original Findings

Based on the repository exploration:

### Key Findings from Original Work:
1. **Emergent World Representation**: GPT trained on Othello develops internal representation of board state
2. **Nonlinear Probes**: 2-layer MLP probes can decode board state with ~1.7% error on synthetic model (vs ~20% for linear probes)
3. **Causal Role**: Interventional experiments show the representation causally affects predictions
4. **Identified Neurons**: Specific neurons like L5N1393 detect specific board configurations (e.g., C0=blank & D1=theirs & E2=mine)

### Models Used in Original Work:
- 8-layer GPT with 8-head attention, 512-dim hidden space
- Trained on synthetic dataset (20M randomly generated legal games)
- Also trained on championship dataset (140K expert games)

### Methodology:
- Linear and nonlinear probes trained on internal activations to predict board state
- Interventional experiments modify activations using probe directions
- Latent saliency maps show which tiles causally contribute to predictions

---

# Generalizability Evaluation

Now we evaluate whether these findings generalize beyond the original experimental setting.

In [4]:
# Set up environment for generalization testing
import os
import sys
import torch
import numpy as np
import json
from pathlib import Path
from copy import deepcopy

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Add repo paths
REPO_ROOT = Path("/net/scratch2/smallyan/othello-world_eval")
sys.path.insert(0, str(REPO_ROOT / "mechanistic_interpretability"))
sys.path.insert(0, str(REPO_ROOT / "data"))

# Set environment for HuggingFace
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_cache'
os.makedirs('/tmp/hf_cache', exist_ok=True)

# Create evaluation directory
eval_dir = REPO_ROOT / "evaluation"
eval_dir.mkdir(exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Evaluation directory: {eval_dir}")

Using device: cuda
Repository root: /net/scratch2/smallyan/othello-world_eval
Evaluation directory: /net/scratch2/smallyan/othello-world_eval/evaluation


In [5]:
# Install required packages
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "transformer_lens==1.2.1", "-q"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "einops", "-q"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/neelnanda-io/neel-plotly.git", "-q"], capture_output=True)
print("Packages installed")

Packages installed


In [6]:
# Import libraries
import torch
import torch.nn as nn
import einops
import transformer_lens
import transformer_lens.utils as tl_utils
from transformer_lens import HookedTransformer, HookedTransformerConfig

# Import utility functions from the repo
from mech_interp_othello_utils import (
    OthelloBoardState, 
    to_string, to_int, 
    int_to_label, string_to_label,
    stoi_indices
)

# Disable gradients for inference
torch.set_grad_enabled(False)
print("Libraries imported successfully")

Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Libraries imported successfully


## GT1: Model Generalization Test

**Goal**: Evaluate whether the neuron-level findings transfer to a **new model** not used in the original work.

The original work used:
- Synthetic model (trained on 20M randomly generated games)
- Championship model (trained on 140K expert games)

For GT1, we need to test on a model NOT used in the original work. Options:
1. Test if findings from synthetic model transfer to championship model (different training data)
2. Test on a differently-sized model if available

**Key finding to test**: Neuron L5N1393 detects specific board configuration (C0=blank & D1=theirs & E2=mine)

In [7]:
# Load the SYNTHETIC model (original model used in the study)
model_config = HookedTransformerConfig(
    n_layers=8,
    d_model=512,
    d_head=64,
    n_heads=8,
    d_mlp=2048,
    d_vocab=61,
    n_ctx=59,
    act_fn="gelu",
    normalization_type="LNPre"
)

synthetic_model = HookedTransformer(model_config)

# Load synthetic model weights
from huggingface_hub import hf_hub_download
synthetic_path = hf_hub_download(
    repo_id="NeelNanda/Othello-GPT-Transformer-Lens", 
    filename="synthetic_model.pth",
    cache_dir='/tmp/hf_cache'
)
synthetic_state_dict = torch.load(synthetic_path, map_location='cuda', weights_only=False)
synthetic_model.load_state_dict(synthetic_state_dict)
synthetic_model = synthetic_model.cuda()

print(f"Synthetic model loaded successfully")
print(f"Parameters: {sum(p.numel() for p in synthetic_model.parameters()):,}")

synthetic_model.pth:   0%|          | 0.00/101M [00:00<?, ?B/s]